In [0]:
--Create silver schema
CREATE SCHEMA IF NOT EXISTS weather_openmeteo.silver;

USE weather_openmeteo.silver;

--Create the daily weather cleaned table 
CREATE TABLE IF NOT EXISTS weather_openmeteo.silver.weather_daily_clean (
    date TIMESTAMP,
    latitude DOUBLE,
    longitude DOUBLE,
    city STRING,
    temperature_2m_max DOUBLE,
    temperature_2m_min DOUBLE,
    daylight_duration DOUBLE,
    wind_speed_10m_max DOUBLE
)
USING DELTA;

-- 1. Deduplicate, validate, and clean the raw daily Bronze data using a CTE
WITH source_cleaned AS (
    SELECT 
        CAST(date AS TIMESTAMP) AS date,
        CAST(latitude AS DOUBLE) AS latitude,
        CAST(longitude AS DOUBLE) AS longitude,
        -- Cleanse temperature: remove outliers and ensure min <= max
        CASE 
            WHEN temperature_2m_max < -90 OR temperature_2m_max > 60 THEN NULL
            ELSE CAST(temperature_2m_max AS DOUBLE)
        END AS temperature_2m_max,
        CASE 
            WHEN temperature_2m_min < -90 OR temperature_2m_min > 60 THEN NULL
            ELSE CAST(temperature_2m_min AS DOUBLE)
        END AS temperature_2m_min,
        -- Cleanse daylight_duration: must be between 0 and 86400 seconds (24h)
        CASE 
            WHEN daylight_duration < 0 OR daylight_duration > 86400 THEN NULL
            ELSE CAST(daylight_duration AS DOUBLE)
        END AS daylight_duration,
        -- Cleanse wind_speed_10m_max: must be non-negative and reasonable (e.g., < 150 m/s)
        CASE 
            WHEN wind_speed_10m_max < 0 OR wind_speed_10m_max > 150 THEN NULL
            ELSE CAST(wind_speed_10m_max AS DOUBLE)
        END AS wind_speed_10m_max,
        -- Conditional mapping logic to assign city names based on coordinates
        CASE 
            WHEN ROUND(latitude, 4) = 42.4955 AND ROUND(longitude, 4) = -89.0273 THEN 'Beloit'
            WHEN ROUND(latitude, 4) = 42.6831 AND ROUND(longitude, 4) = -89.0038 THEN 'Janesville'
            WHEN ROUND(latitude, 4) = 43.0604 AND ROUND(longitude, 4) = -89.3995 THEN 'Madison'
            ELSE 'Unknown'
        END AS city,
        ROW_NUMBER() OVER(
            PARTITION BY date, latitude, longitude 
            ORDER BY date DESC
        ) as row_num
    FROM weather_openmeteo.bronze.weather_daily
    -- Filter out records containing NULL values in key columns
    WHERE date IS NOT NULL 
      AND latitude IS NOT NULL 
      AND longitude IS NOT NULL
      -- Remove records with obviously invalid coordinates
      AND latitude BETWEEN -90 AND 90
      AND longitude BETWEEN -180 AND 180
      -- Remove records with NULLs in metrics
      AND temperature_2m_max IS NOT NULL
      AND temperature_2m_min IS NOT NULL
      AND daylight_duration IS NOT NULL
      AND wind_speed_10m_max IS NOT NULL
)

, validated AS (
    SELECT
        date,
        latitude,
        longitude,
        city,
        temperature_2m_max,
        temperature_2m_min,
        daylight_duration,
        wind_speed_10m_max,
        row_num
    FROM source_cleaned
    -- Remove records where cleansing resulted in NULLs in any metric
    WHERE temperature_2m_max IS NOT NULL
      AND temperature_2m_min IS NOT NULL
      AND daylight_duration IS NOT NULL
      AND wind_speed_10m_max IS NOT NULL
      -- Ensure temperature min is not greater than max
      AND temperature_2m_min <= temperature_2m_max
)

-- 2. Execute the idempotent Merge operation into the Silver table
MERGE INTO weather_daily_clean AS target
USING (
    SELECT date, latitude, longitude, city, temperature_2m_max, temperature_2m_min, daylight_duration, wind_speed_10m_max 
    FROM validated 
    WHERE row_num = 1
) AS source
ON target.date = source.date 
   AND target.latitude = source.latitude 
   AND target.longitude = source.longitude

-- When the record already exists, update the mutable metrics and attributes
WHEN MATCHED THEN
  UPDATE SET
    target.city = source.city,
    target.temperature_2m_max = source.temperature_2m_max,
    target.temperature_2m_min = source.temperature_2m_min,
    target.daylight_duration = source.daylight_duration,
    target.wind_speed_10m_max = source.wind_speed_10m_max

-- When the record is new, insert all columns into the target table
WHEN NOT MATCHED THEN
  INSERT (date, latitude, longitude, city, temperature_2m_max, temperature_2m_min, daylight_duration, wind_speed_10m_max)
  VALUES (source.date, source.latitude, source.longitude, source.city, source.temperature_2m_max, source.temperature_2m_min, source.daylight_duration, source.wind_speed_10m_max);

--Show the Silver table
SELECT * FROM weather_daily_clean;